# 05 - Prompt Construction
**Module:** `src/prompts/templates.py`

## What this module does
Constructs the exact message list sent to Groq for every lesson generation.
A well-engineered prompt is the difference between a lesson that follows
all our blueprint rules and one that drifts into something generic.

## The three-part prompt structure
1. **System prompt** — who the model is, what rules it follows, output format
2. **Few-shot example** — one hand-crafted lesson shown as a reference
3. **User prompt** — the specific grade band + domain + theme request

## Why few-shot examples?
Showing the model a complete example at the right grade band dramatically
reduces hallucination risk. This is our lightweight substitute for RAG
(Retrieval Augmented Generation) -- we get the grounding benefit without
needing a vector database.

## What we'll explore
1. The system prompt — what the model reads as its instructions
2. The few-shot example — which lesson gets selected per grade band
3. The user prompt — the actual generation request
4. The full assembled message list
5. Token estimation across grade bands

In [1]:
import sys
sys.path.insert(0, '..')

from src.prompts.templates import (
    build_system_prompt,
    build_user_prompt,
    build_full_prompt,
    inspect_prompt,
    SCHEMA_DESCRIPTION,
)
from src.core.skill_selector import get_next_skill


In [2]:
# The Schema Description
# This is the JSON format specification injected into every system prompt.
# It tells the model exactly what fields to produce and in what structure.

print(SCHEMA_DESCRIPTION)

You must return a single valid JSON object with this exact structure:

{
  "lesson_id": "",
  "metadata": {
    "grade_band": "one of: K-2 | 3-5 | 6-8 | 9-12",
    "ela_domain": "one of: Speaking | Listening | Reading | Writing | Reading → Speaking",
    "lesson_type": "one of: Story Retell | Mission Brief | Debate Drop | Text Explorer | Listen & Judge",
    "theme": "the theme you were given",
    "primary_skill": "use the exact skill string you were given — do not modify it",
    "voice_markers": ["1 or 2 from: Pronunciation & Articulation | Prosody | Speaking Rate | Fluency & Fillers | Volume Control | Task Adherence"],
    "estimated_duration_minutes": integer between 4 and 10,
    "ccss_anchor": "the CCSS standard this lesson maps to",
    "design_notes": "brief note on your design decisions"
  },
  "lesson_flow": {
    "hook": {
      "duration_seconds": integer,
      "content": "the narrative scene-setting text"
    },
    "model": {
      "duration_seconds": integer,
      "co

# 1. The System Prompt
The system prompt combines:
- The role definition (who the model is)
- The 5 lesson design principles with research citations
- The grade band spec (injected dynamically)
- The JSON schema the model must return

Let's look at it for K-2 first — the most constrained grade band.

In [3]:
# Build and print the K-2 system prompt
system_prompt = build_system_prompt("K-2")
print(system_prompt)

You are an expert K-12 ELA lesson designer for Bantrly, a voice-based
language learning platform for students. Your job is to generate a single,
complete, structured lesson in JSON format.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
LESSON DESIGN PRINCIPLES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Every lesson you generate must follow these principles:

1. SINGLE SKILL: Each lesson teaches exactly ONE primary skill.
   A lesson that teaches multiple skills teaches nothing measurable.

2. NARRATIVE FIRST: Every lesson wraps its skill practice in a narrative
   frame — a character, a problem, a mission, or a scenario.
   Students need something to say before they can practice saying it well.
   (Bruner, 1990 — narrative mode of cognition)

3. FOUR-STAGE FLOW: Hook → Model → Practice → Reflect.
   - Hook: Set the scene. Activate curiosity. No skill instruction yet.
   - Model: Show what the skill sounds like. Name it explicitly.
   - Practice: Student speaks. 1-3 prompts, scaffolded → inde

In [4]:
# Compare system prompt lengths across grade bands
print("System prompt lengths by grade band:\n")
for band in ["K-2", "3-5", "6-8", "9-12"]:
    prompt = build_system_prompt(band)
    tokens = len(prompt) // 4  # rough token estimate
    print(f"  {band}: {len(prompt):,} chars (~{tokens:,} tokens)")

System prompt lengths by grade band:

  K-2: 6,555 chars (~1,638 tokens)
  3-5: 6,560 chars (~1,640 tokens)
  6-8: 6,693 chars (~1,673 tokens)
  9-12: 6,712 chars (~1,678 tokens)


## Part 3: The Few-Shot Example
The second message in our prompt is a complete hand-crafted lesson.
The model studies this before generating its own.

Notice: the example is selected by grade band -- a K-2 request
gets a K-2 example, not a 9-12 one. This keeps the complexity
level of the reference appropriate for the target output.

In [5]:
import json
from src.utils.file_handler import load_example_by_grade

# Show which example gets selected for each grade band
print("Few-shot example selected per grade band:\n")
for band in ["K-2", "3-5", "6-8", "9-12"]:
    example = load_example_by_grade(band)
    print(f"  {band} → {example['lesson_id']}")
    print(f"       Theme : {example['metadata']['theme']}")
    print(f"       Type  : {example['metadata']['lesson_type']}")
    print(f"       Skill : {example['metadata']['primary_skill'][:60]}...")
    print()

Few-shot example selected per grade band:

  K-2 → L-K2-SPK-001
       Theme : Nature & Animals
       Type  : Story Retell
       Skill : Retell the key events of a short story in sequence using beg...

  3-5 → L-35-SPK-005
       Theme : Adventure & Discovery
       Type  : Mission Brief
       Skill : Explain a process clearly using step-by-step sequence and re...

  6-8 → L-68-SPK-003
       Theme : Ethics & Dilemmas
       Type  : Debate Drop
       Skill : Present a claim with at least two pieces of supporting evide...

  9-12 → L-912-RDG-SPK-004
       Theme : History & Change
       Type  : Text Explorer
       Skill : Analyze an author's rhetorical choices and explain how they ...



## Part 4: The User Prompt
The final message -- short and direct.
All the complexity lives in the system prompt and few-shot example.
The user prompt just specifies what to generate.

In [6]:
# Build user prompts for different requests
requests = [
    ("K-2",  "Speaking",          "Nature & Animals"),
    ("3-5",  "Speaking",          "Space Exploration"),
    ("6-8",  "Speaking",          "Climate Change"),
    ("9-12", "Reading → Speaking", "Artificial Intelligence Ethics"),
]

for grade, domain, theme in requests:
    skill  = get_next_skill(grade, domain)
    prompt = build_user_prompt(grade, domain, theme, skill)
    print(f"=== {grade} | {domain} | {theme} ===")
    print(prompt)
    print()

=== K-2 | Speaking | Nature & Animals ===
Generate a complete Bantrly lesson with the following parameters:

  Grade Band     : K-2
  ELA Domain     : Speaking
  Theme          : Nature & Animals
  Primary Skill  : Retell a story in sequence using beginning, middle, and end

IMPORTANT: You must use the exact Primary Skill string above as the 'primary_skill' field in your JSON. Do not modify, rephrase, or replace it. Build the entire lesson around this specific skill.

Follow all grade band rules and design principles exactly. Return only the JSON object.

=== 3-5 | Speaking | Space Exploration ===
Generate a complete Bantrly lesson with the following parameters:

  Grade Band     : 3-5
  ELA Domain     : Speaking
  Theme          : Space Exploration
  Primary Skill  : Explain a process step by step in your own words

IMPORTANT: You must use the exact Primary Skill string above as the 'primary_skill' field in your JSON. Do not modify, rephrase, or replace it. Build the entire lesson aro

## Part 5: The Full Message List
This is what actually gets sent to Groq -- all four messages assembled.
Let's use `inspect_prompt()` to see the structure without dumping
the entire few-shot example JSON.

In [7]:
# Build and inspect the full prompt for a 6-8 Speaking lesson
skill = get_next_skill("6-8", "Speaking")
messages = build_full_prompt("6-8", "Speaking", "Climate Change", skill)
print(f"Skill injected: {skill}\n")
inspect_prompt(messages)

Skill injected: Present a claim and support it with two pieces of evidence

Total messages: 4

[1] SYSTEM
----------------------------------------
You are an expert K-12 ELA lesson designer for Bantrly, a voice-based
language learning platform for students. Your job is to generate a single,
complete, structured lesson in JSON format.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
LESSON DESIGN PRINCIPLES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Ev...

[2] USER
----------------------------------------
Here is a complete example of a well-designed Bantrly lesson for grade band 6-8. Study its structure, depth, and how it follows all the design principles:

{
  "lesson_id": "L-68-SPK-003",
  "metadata": {
    "grade_band": "6-8",
    "ela_domain": "Speaking",
    "lesson_type": "Debate Drop",
    "t...

[3] ASSISTANT
----------------------------------------
Understood. I have studied the example lesson carefully. I will follow the same structure, depth, narrative quality, and JSON format exact

In [8]:
# Compare total prompt sizes across grade bands
print("Total prompt size by grade band:\n")
print(f"  {'Grade':<8} {'Chars':>10} {'Est. Tokens':>14}")
print(f"  {'-'*8:<8} {'-'*10:>10} {'-'*14:>14}")

for band in ["K-2", "3-5", "6-8", "9-12"]:
    skill       = get_next_skill(band, "Speaking")
    messages = build_full_prompt(band, "Speaking", "Test Theme", skill)
    total_chars = sum(len(m["content"]) for m in messages)
    est_tokens  = total_chars // 4
    print(f"  {band:<8} {total_chars:>10,} {est_tokens:>14,}")

Total prompt size by grade band:

  Grade         Chars    Est. Tokens
  -------- ---------- --------------
  K-2          10,852          2,713
  3-5          11,651          2,912
  6-8          11,862          2,965
  9-12         12,348          3,087


## Part 6: Reading the Prompt Like the Model Does
Let's print the full prompt for K-2 exactly as the model receives it --
message by message. This shows precisely what instructions the model is working from.

In [9]:
# Print the full K-2 prompt — every message in full
skill       = get_next_skill("K-2", "Speaking")
messages = build_full_prompt("K-2", "Speaking", "Nature & Animals", skill)

for i, msg in enumerate(messages):
    role = msg["role"].upper()
    print(f"\n{'='*60}")
    print(f"MESSAGE {i+1}: {role}")
    print('='*60)
    print(msg["content"])


MESSAGE 1: SYSTEM
You are an expert K-12 ELA lesson designer for Bantrly, a voice-based
language learning platform for students. Your job is to generate a single,
complete, structured lesson in JSON format.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
LESSON DESIGN PRINCIPLES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Every lesson you generate must follow these principles:

1. SINGLE SKILL: Each lesson teaches exactly ONE primary skill.
   A lesson that teaches multiple skills teaches nothing measurable.

2. NARRATIVE FIRST: Every lesson wraps its skill practice in a narrative
   frame — a character, a problem, a mission, or a scenario.
   Students need something to say before they can practice saying it well.
   (Bruner, 1990 — narrative mode of cognition)

3. FOUR-STAGE FLOW: Hook → Model → Practice → Reflect.
   - Hook: Set the scene. Activate curiosity. No skill instruction yet.
   - Model: Show what the skill sounds like. Name it explicitly.
   - Practice: Student speaks. 1-3 prompts

## Part 7: Previewing Without Making an API Call
The `preview_prompt()` method on `LessonGenerator` lets you inspect
the prompt interactively before spending any API tokens.
This is useful for debugging prompt issues without burning API quota.

In [11]:
from src.core.generator import LessonGenerator

gen = LessonGenerator(verbose=False)

# Preview the prompt for a 9-12 lesson — no API call made
skill = get_next_skill("9-12", "Speaking")
print("Previewing 9-12 prompt — no API call:\n")
print(f"Skill that will be injected: {skill}\n")
gen.preview_prompt("9-12", "Speaking", "Artificial Intelligence Ethics")

Previewing 9-12 prompt — no API call:

Skill that will be injected: Construct and deliver a persuasive argument using logos, ethos, and pathos

[preview] Auto-selected skill: Construct and deliver a persuasive argument using logos, ethos, and pathos

Total messages: 4

[1] SYSTEM
----------------------------------------
You are an expert K-12 ELA lesson designer for Bantrly, a voice-based
language learning platform for students. Your job is to generate a single,
complete, structured lesson in JSON format.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
LESSON DESIGN PRINCIPLES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Ev...

[2] USER
----------------------------------------
Here is a complete example of a well-designed Bantrly lesson for grade band 9-12. Study its structure, depth, and how it follows all the design principles:

{
  "lesson_id": "L-912-RDG-SPK-004",
  "metadata": {
    "grade_band": "9-12",
    "ela_domain": "Reading \u2192 Speaking",
    "lesson_type":...

[3] ASSISTANT
----

## Summary

| What we explored | Key takeaway |
|---|---|
| Schema description | Exact JSON format the model must produce — injected into every prompt |
| System prompt | Combines role + design principles + grade spec + schema |
| Few-shot example | Grade-matched example grounds the model at the right complexity |
| User prompt | Now includes the taxonomy skill — LLM cannot invent or deviate from it |
| Full message list | 4 messages, ~2,500-2,900 tokens total — efficient for free tier |
| `preview_prompt()` | Auto-selects next skill from taxonomy — same behaviour as `generate()` |

**Design decision this validates:**
The few-shot example is the single most important prompt engineering
decision in this system. Without it, the model produces structurally
correct JSON but lessons that drift from the blueprint. With it,
the model has a concrete reference for what "good" looks like
at the right grade band.

Injecting the skill directly into the user prompt was added after
observing that the LLM produced inconsistent skill strings when
selecting freely -- even with grade band rules in the system prompt.
Explicit injection eliminated the problem entirely.

**Known limitation:**
We use one few-shot example per grade band. A production system
would retrieve the most relevant example by theme similarity
using embeddings -- true RAG. `preview_prompt()` currently shows
the next uncovered skill. In production it would also accept a
student ID to preview a personalised skill selection.